# 01 · EDA on historical Citi Bike trip dataWhat this notebook establishes before any modelling:1. How much data there is and what shape it has.2. That the **join key** between trip CSVs and GBFS is `short_name`, not `station_id`.3. Why the target is aggregated to **clusters** rather than modelled per station.4. Which signals actually drive demand — and therefore which features are worth building.Prerequisite: `python -m scripts.bootstrap_db` or a prior training run, so theparquet cache under `data/cache/` exists.

In [ ]:
import warningswarnings.filterwarnings("ignore")import matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom src.config import data_dir, load_configconfig = load_config()departures = pd.read_parquet(data_dir("cache", "hourly_departures_all.parquet"))stations = pd.read_parquet(data_dir("cache", "station_information.parquet"))departures["hour_ts"] = pd.to_datetime(departures["hour_ts"])print(f"{len(departures):,} station-hour rows")print(f"{departures['station_short_name'].nunique():,} stations")print(f"{departures['hour_ts'].min()} -> {departures['hour_ts'].max()}")print(f"{int(departures['departures'].sum()):,} total departures")

## The join keyModern trip CSVs carry `start_station_id` values like `5506.14`. Those match theGBFS **`short_name`** field, *not* GBFS `station_id`, which is a UUID(`66dd1f44-0aca-11e7-...`). Joining on `station_id` silently matches zero rows —worth verifying explicitly rather than assuming.

In [ ]:
trip_ids = set(departures["station_short_name"].unique())by_short_name = len(trip_ids & set(stations["short_name"].dropna()))by_station_id = len(trip_ids & set(stations["station_id"].dropna()))print(f"matched on short_name : {by_short_name:,} / {len(trip_ids):,}")print(f"matched on station_id : {by_station_id:,} / {len(trip_ids):,}   <- the wrong key")

## Why not model per stationThe per-station hourly target is dominated by zeros and small counts, which makesit noisy and makes any error metric hard to interpret. Clustering pools the signal.

In [ ]:
per_station = departures.groupby("station_short_name")["departures"].agg(["sum", "mean", "median"])print(per_station.describe().round(2))zero_share = (departures["departures"] == 0).mean()print(f"\nShare of observed station-hours with zero departures: {zero_share:.1%}")print("(and that excludes station-hours absent from the data entirely, which are also zeros)")fig, ax = plt.subplots(figsize=(9, 4))ax.hist(per_station["mean"], bins=60)ax.set_xlabel("mean departures per hour, per station")ax.set_ylabel("stations")ax.set_title("Most stations see only a handful of departures an hour")plt.tight_layout()

In [ ]:
from src.features.station_clusters import aggregate_to_entities, fit_entity_mappingmapping = fit_entity_mapping(stations, config, stations_in_scope=trip_ids)entity_hourly = aggregate_to_entities(departures, mapping)print(f"{len(mapping.entities)} clusters, median {int(mapping.entities['n_stations'].median())} stations each")print(f"mean departures per cluster-hour : {entity_hourly['departures'].mean():.1f}")print(f"zero share after clustering      : {(entity_hourly['departures'] == 0).mean():.2%}")

## Demand driversHour-of-day and day-of-week dominate, which is why the feature set leans oncyclical time encodings plus same-hour-last-week lags.

In [ ]:
frame = entity_hourly.copy()frame["hour"] = frame["hour_ts"].dt.hourframe["dow"] = frame["hour_ts"].dt.dayofweekframe["is_weekend"] = frame["dow"] >= 5fig, axes = plt.subplots(1, 2, figsize=(14, 4))for weekend, group in frame.groupby("is_weekend"):    profile = group.groupby("hour")["departures"].mean()    axes[0].plot(profile.index, profile.values, marker="o",                 label="weekend" if weekend else "weekday")axes[0].set(xlabel="hour of day", ylabel="mean departures per cluster",            title="Commute peaks on weekdays, a midday hump at weekends")axes[0].legend()frame.groupby("dow")["departures"].mean().plot(kind="bar", ax=axes[1])axes[1].set(xlabel="day of week (0=Mon)", ylabel="mean departures", title="Day-of-week effect")plt.tight_layout()

In [ ]:
from src.ingestion.weather import weather_window_forweather = weather_window_for(entity_hourly["hour_ts"], config)merged = entity_hourly.merge(weather, on="hour_ts", how="left")city = merged.groupby("hour_ts").agg(    departures=("departures", "sum"),    temperature_c=("temperature_c", "first"),    precipitation_mm=("precipitation_mm", "first"),).dropna()print("Correlation with system-wide hourly demand:")print(city[["departures", "temperature_c", "precipitation_mm"]].corr()["departures"].round(3))dry = city[city["precipitation_mm"] < 0.1]["departures"].mean()wet = city[city["precipitation_mm"] >= 1.0]["departures"].mean()print(f"\nmean demand, dry hours : {dry:,.0f}")print(f"mean demand, rainy hours: {wet:,.0f}   ({wet/dry - 1:+.1%})")

## Takeaways that shaped the model- **`short_name` is the join key.** Verified above.- **Cluster granularity**, because the per-station target is mostly zeros.- **Cyclical hour/day-of-week encodings**, since the daily profile is the strongest signal.- **Same-hour-last-week lag (168h)** as the single most informative lag — and the  seasonal-naive baseline every model is measured against.- **Weather is worth the extra dependency**: rain depresses demand materially.- **Only Apr–Jun is covered**, so the model has never seen winter. Recorded as a  limitation in the README rather than glossed over.